In [ ]:
import asyncio
import json
import random
import re
from dataclasses import asdict, dataclass
from datetime import datetime
from typing import Any, Dict, List, Optional
from urllib.parse import urlencode, urljoin

import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import Page, async_playwright
from tqdm import tqdm


# -----------------------------
# Config / Utils
# -----------------------------
def clean_text(s: Optional[str]) -> Optional[str]:
    if not s:
        return None
    s = re.sub(r"\s+", " ", s).strip()
    return s or None


def parse_int_like(s: Optional[str]) -> Optional[int]:
    """Extract integer from strings like '1,234', '좋아요 12', '12'."""
    if not s:
        return None
    m = re.search(r"(\d[\d,]*)", s)
    if not m:
        return None
    return int(m.group(1).replace(",", ""))


def now_iso():
    return datetime.now().isoformat(timespec="seconds")


@dataclass
class DaangnRow:
    keyword: str
    title: Optional[str]
    price: Optional[str]
    region: Optional[str]

    status: Optional[str]          # 예약중 / 거래완료 / (없음)
    posted_at: Optional[str]       # 올라온 날짜/시간(또는 "3분 전")
    bumped_at: Optional[str]       # 끌올 시간(있으면)
    is_bumped: Optional[bool]

    image_count: Optional[int]
    like_count: Optional[int]
    message_count: Optional[int]   # 공개로 표시될 때만

    url: str
    crawled_at: str


# -----------------------------
# DOM Parsing (IMPORTANT: tune selectors)
# -----------------------------
def extract_from_detail_html(html: str) -> Dict[str, Any]:
    """
    Extract fields from a listing detail page HTML.
    You will likely need to adjust selectors for:
      - title
      - price
      - region/time
      - status (예약중/거래완료)
      - likes/messages (if shown)
      - image count

    Strategy:
    - Use robust heuristics from text and common structural patterns.
    """
    soup = BeautifulSoup(html, "lxml")

    text_all = clean_text(soup.get_text(" ", strip=True)) or ""

    # ---- Title (try common candidates)
    title = None
    # heuristic: first h1
    h1 = soup.find("h1")
    if h1:
        title = clean_text(h1.get_text(" ", strip=True))
    if not title:
        # fallback: meta
        ogt = soup.find("meta", {"property": "og:title"})
        if ogt and ogt.get("content"):
            title = clean_text(ogt["content"])

    # ---- Price
    price = None
    # search for "원" price-like tokens
    m_price = re.search(r"(\d[\d,]*\s*원)", text_all)
    if m_price:
        price = clean_text(m_price.group(1))

    # ---- Status: 예약중 / 거래완료 (if visible as badge text)
    status = None
    for key in ["예약중", "거래완료", "판매완료"]:
        if key in text_all:
            # "판매완료" is sometimes used; normalize to 거래완료 if you want
            status = "거래완료" if key in ["거래완료", "판매완료"] else "예약중"
            break

    # ---- Posted time/date (often shown like "3분 전", "1시간 전", "2일 전")
    posted_at = None
    m_time = re.search(r"(\d+\s*(분|시간|일)\s*전)", text_all)
    if m_time:
        posted_at = clean_text(m_time.group(1))
    else:
        # sometimes formatted dates exist; this is a very rough fallback
        m_date = re.search(r"(20\d{2}[./-]\d{1,2}[./-]\d{1,2})", text_all)
        if m_date:
            posted_at = clean_text(m_date.group(1))

    # ---- Bumped (끌올) time
    # Some pages show "끌올" or "끌어올림" text.
    bumped_at = None
    is_bumped = None
    if "끌올" in text_all or "끌어올림" in text_all:
        is_bumped = True
        # try to find "끌올" near a "n분 전" token (rough)
        m_bump = re.search(r"(끌올|끌어올림).{0,20}?(\d+\s*(분|시간|일)\s*전)", text_all)
        if m_bump:
            bumped_at = clean_text(m_bump.group(2))
    else:
        is_bumped = False

    # ---- Image count
    # Heuristic: count unique <img> in a likely gallery area
    # NOTE: There may be many UI icons. We'll prefer og:image? Not enough.
    imgs = soup.find_all("img")
    # Filter out tiny/icon images by checking attributes
    img_urls = []
    for img in imgs:
        src = img.get("src") or img.get("data-src") or ""
        src = src.strip()
        if not src:
            continue
        # filter obvious sprites/icons
        if any(x in src.lower() for x in ["sprite", "icon", "logo", "data:image"]):
            continue
        img_urls.append(src)
    # Dedup
    image_count = len(set(img_urls)) if img_urls else None

    # ---- Like count (if visible)
    like_count = None
    # common text patterns: "관심 12", "좋아요 12"
    m_like = re.search(r"(관심|좋아요)\s*(\d[\d,]*)", text_all)
    if m_like:
        like_count = parse_int_like(m_like.group(2))

    # ---- Message count (if visible)
    message_count = None
    # common text patterns: "채팅 3", "메시지 3"
    m_msg = re.search(r"(채팅|메시지)\s*(\d[\d,]*)", text_all)
    if m_msg:
        message_count = parse_int_like(m_msg.group(2))

    # ---- Region (rough heuristic: "*구" / "*동")
    region = None
    m_region = re.search(r"([가-힣]{2,}(구|동))", text_all)
    if m_region:
        region = clean_text(m_region.group(1))

    return {
        "title": title,
        "price": price,
        "region": region,
        "status": status,
        "posted_at": posted_at,
        "bumped_at": bumped_at,
        "is_bumped": is_bumped,
        "image_count": image_count,
        "like_count": like_count,
        "message_count": message_count,
    }


def extract_listing_links_from_search_html(html: str, base_url: str) -> List[str]:
    """
    Extract listing links from a search result page.
    This MUST be tuned to the actual link pattern you see.
    """
    soup = BeautifulSoup(html, "lxml")
    links = []

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if not href or href.startswith("#"):
            continue

        # ---- IMPORTANT: adjust these patterns based on actual Daangn URLs you see
        if ("/articles/" in href) or ("/buy-sell/" in href) or ("/products/" in href):
            links.append(urljoin(base_url, href))

    # de-dup while preserving order
    seen = set()
    out = []
    for u in links:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


# -----------------------------
# Playwright crawling
# -----------------------------
async def polite_sleep(min_s: float, max_s: float):
    await asyncio.sleep(random.uniform(min_s, max_s))


async def get_html(page: Page, url: str) -> str:
    await page.goto(url, wait_until="domcontentloaded", timeout=60_000)
    # allow lazy-load
    await page.wait_for_timeout(800)
    return await page.content()


def build_search_urls(keyword: str, page_limit: int) -> List[str]:
    """
    We avoid guessing Daangn internal search API.
    Recommended approach:
      - You provide a REAL search URL from your browser as start_url, and we paginate it.
    But if you want keyword-only mode, we can build a generic query URL if known.

    For safety/robustness, this returns placeholders and expects start_url usage.
    """
    raise NotImplementedError(
        "Use `start_url` (copy from browser) mode below. "
        "Daangn search URLs vary and change; keyword-only URL building is not stable."
    )


def paginate_url(start_url: str, page_idx: int) -> str:
    """
    Generic pagination:
    - if start_url already has page=, replace it
    - else append page=
    Adjust if Daangn uses cursor-based pagination.
    """
    if "page=" in start_url:
        return re.sub(r"page=(\d+)", f"page={page_idx}", start_url)
    sep = "&" if "?" in start_url else "?"
    return f"{start_url}{sep}page={page_idx}"


async def crawl_daangn(
    *,
    keyword: str,
    start_url: str,
    page_limit: int = 3,
    item_limit: int = 50,
    headless: bool = True,
    min_delay: float = 1.2,
    max_delay: float = 2.2,
    out_csv: str = "daangn_polo.csv",
    out_json: str = "daangn_polo.json",
):
    """
    Public-page crawler template. You must provide a real `start_url`:
      - Go to Daangn in your browser
      - search "폴로"
      - copy the URL of the results page
      - paste it to start_url

    page_limit: how many result pages to visit
    item_limit: max number of listings to process in total
    """
    # base URL for urljoin
    m = re.match(r"^(https?://[^/]+)/", start_url)
    if not m:
        raise ValueError("start_url must be a full URL like https://...")

    base_url = m.group(1) + "/"

    rows: List[DaangnRow] = []
    seen_links = set()

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=headless)
        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122 Safari/537.36"
            )
        )
        page = await context.new_page()

        # 1) Collect links from search pages
        all_links: List[str] = []
        for page_idx in range(1, page_limit + 1):
            url = paginate_url(start_url, page_idx)
            html = await get_html(page, url)
            links = extract_listing_links_from_search_html(html, base_url=base_url)

            # add new
            added = 0
            for u in links:
                if u not in seen_links:
                    seen_links.add(u)
                    all_links.append(u)
                    added += 1
                if len(all_links) >= item_limit:
                    break

            print(f"[Search page {page_idx}/{page_limit}] found={len(links)}, added={added}, total_links={len(all_links)}")

            if len(all_links) >= item_limit:
                break

            await polite_sleep(min_delay, max_delay)

        # 2) Visit each listing detail page and extract fields
        pbar = tqdm(all_links, desc="Crawling listings", unit="item")
        for u in pbar:
            try:
                html = await get_html(page, u)
                info = extract_from_detail_html(html)

                row = DaangnRow(
                    keyword=keyword,
                    title=info.get("title"),
                    price=info.get("price"),
                    region=info.get("region"),
                    status=info.get("status"),
                    posted_at=info.get("posted_at"),
                    bumped_at=info.get("bumped_at"),
                    is_bumped=info.get("is_bumped"),
                    image_count=info.get("image_count"),
                    like_count=info.get("like_count"),
                    message_count=info.get("message_count"),
                    url=u,
                    crawled_at=now_iso(),
                )
                rows.append(row)

                # show a short progress postfix
                pbar.set_postfix(
                    title=(row.title[:12] + "…") if row.title else None,
                    status=row.status,
                    likes=row.like_count,
                    imgs=row.image_count,
                )

            except Exception as e:
                pbar.set_postfix(error=str(e)[:60])

            await polite_sleep(min_delay, max_delay)

        await browser.close()

    # Save
    df = pd.DataFrame([asdict(r) for r in rows])

    # Column order
    cols = [
        "keyword", "title", "price", "region",
        "status", "posted_at", "bumped_at", "is_bumped",
        "image_count", "like_count", "message_count",
        "url", "crawled_at",
    ]
    df = df.reindex(columns=cols)

    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(df.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

    print(f"\nSaved CSV: {out_csv} ({len(df)} rows)")
    print(f"Saved JSON: {out_json}")
    return df


if __name__ == "__main__":
    # 1) 브라우저에서 "폴로" 검색 후, 결과 페이지 URL을 그대로 붙여넣기
    START_URL = "PASTE_DAANGN_SEARCH_URL_HERE"

    asyncio.run(
        crawl_daangn(
            keyword="폴로",
            start_url=START_URL,
            page_limit=3,   # 검색 결과 페이지 몇 장 볼지
            item_limit=50,  # 게시물 최대 몇 개 크롤할지
            headless=True,
            min_delay=1.2,  # 너무 빠르면 차단 위험
            max_delay=2.2,
            out_csv="daangn_polo.csv",
            out_json="daangn_polo.json",
        )
    )